# Diabetes Challenge

Your task today is to **analyze** the Kaggle "Pima Indians Diabetes Database" and to **predict** whether a patient has Diabetes or not.

## Task:
- Load the data from the database. The schema is called `diabetes`. To connect to the database you need to copy the `.env` file from the visualization or hands-on-ml repository into this repo. Explore the database, try to establish what the relationships between the tables are (1-1, 1-N, N-M). Explain to yourself and the group what data do you see and whether it makes sense. What JOINs are appropriate to use and why? 
- Use at least two different classification algorithms we have learned so far to predict Diabetes patients. 
- Discuss before you start with the modeling process which **evaluation metric** you choose and explain why.
- Implement a GridSearchCV or RandomizedSearchCV to tune the hyperparameters of your model.
- **Optional:** If you have time at the end, try to use sklearn's pipline module to encapsulate all the steps into a pipeline.

Don't forget to split your data in train and test set. And analyze your final model on the test data. It might also be necessary to scale your data in order to improve the performance of some of the models.


## Helpful links and advise:
- [sklearn documentation on hyperparameter tuning](https://scikit-learn.org/stable/modules/grid_search.html#grid-search)
- It might be helpful to check some sources on how to deal with imbalanced data. 
    * [8 Tactics to Combat Imbalanced Classes](https://machinelearningmastery.com/tactics-to-combat-imbalanced-classes-in-your-machine-learning-dataset/)
    * [Random-Oversampling/Undersampling](https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/)


# Data Description

## Context
This dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

## Acknowledgements
Smith, J.W., Everhart, J.E., Dickson, W.C., Knowler, W.C., & Johannes, R.S. (1988). Using the ADAP learning algorithm to forecast the onset of diabetes mellitus. In Proceedings of the Symposium on Computer Applications and Medical Care (pp. 261--265). IEEE Computer Society Press.

## About this dataset
The datasets consist of several medical predictor (independent) variables and one target (dependent) variable, Outcome. Independent variables include the number of pregnancies the patient has had, their BMI, insulin level, age, and so on. For the outcome class value 1 is interpreted as "tested positive for diabetes".

|Column Name| Description|
|:------------|:------------|
|Pregnancies|Number of times pregnant|
|Glucose|Plasma glucose concentration a 2 hours in an oral glucose tolerance test|
|BloodPressure|Diastolic blood pressure (mm Hg)|
|SkinThickness|Triceps skin fold thickness (mm)|
|Insulin|2-Hour serum insulin (mu U/ml)|
|BMI|Body mass index (weight in kg/(height in m)^2)|
|DiabetesPedigreeFunction| Diabetes pedigree function|
|Age| Age (years)|
|Outcome|Class variable (0 or 1) |

## Step 1 - Goal and Success Metric

Goal: predict whether a patient has diabetes (`Outcome = 1`) from eight medical measurements.

This is a medical screening task. A **false negative** (a diabetic patient told they are healthy) is much more costly than a **false positive** (a healthy patient sent to a second test). Accuracy treats both errors the same, and recall alone would reward a model that flags everyone.

I therefore select and tune models on the **F2 score** (F-beta with beta = 2), which weights recall twice as much as precision.

In [10]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
db = create_engine(os.getenv("DB_STRING"))

tables = pd.read_sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'diabetes';
""", db)
tables

,table_name
0,blood_metrics
1,patient
2,pedigree_outcome
3,skin


In [11]:
db = create_engine(os.getenv("DB_STRING"), pool_pre_ping=True)

for t in tables["table_name"]:
    with db.connect() as conn:
        df_t = pd.read_sql(f"SELECT * FROM diabetes.{t};", conn)
    print(t, df_t.shape)
    print(list(df_t.columns))
    print("-" * 40)

blood_metrics (1771, 6)
['insulin', 'glucose', 'bloodpressure', 'patientid', 'id', 'measurement_date']
----------------------------------------
patient (768, 4)
['id', 'Age', 'pregnancies', 'bmi']
----------------------------------------
pedigree_outcome (768, 4)
['diabetespedigreefunction', 'outcome', 'patientid', 'id']
----------------------------------------
skin (768, 3)
['skinthickness', 'patientid', 'id']
----------------------------------------


In [12]:
with db.connect() as conn:
    blood = pd.read_sql("SELECT * FROM diabetes.blood_metrics;", conn)

print(blood["patientid"].value_counts().value_counts().sort_index())
print(blood["measurement_date"].value_counts())

count
1    235
2    768
Name: count, dtype: int64
measurement_date
2022-12-13    1003
2022-12-01     768
Name: count, dtype: int64


In [13]:
with db.connect() as conn:
    patient = pd.read_sql("SELECT * FROM diabetes.patient;", conn)

blood_ids = set(blood["patientid"])
patient_ids = set(patient["id"])

print("in blood:", len(blood_ids))
print("in patient:", len(patient_ids))
print("in both:", len(blood_ids & patient_ids))
print("only in blood:", len(blood_ids - patient_ids))
print("only in patient:", len(patient_ids - blood_ids))

in blood: 1003
in patient: 768
in both: 768
only in blood: 235
only in patient: 0


In [14]:
real = blood[blood["patientid"].isin(patient["id"])].copy()
real["measurement_date"] = real["measurement_date"].astype(str)

first_id = patient["id"].iloc[0]
print(real[real["patientid"] == first_id].sort_values("measurement_date"))

cols = ["glucose", "insulin", "bloodpressure"]
d1 = real[real["measurement_date"] == "2022-12-01"].set_index("patientid")[cols]
d2 = real[real["measurement_date"] == "2022-12-13"].set_index("patientid")[cols]
d2 = d2.loc[d1.index]

print((d1 == d2).mean())

     insulin  glucose  bloodpressure  patientid   id measurement_date
768       80      172            117          1  769       2022-12-01
0          0      148             72          1    1       2022-12-13
glucose          0.005208
insulin          0.006510
bloodpressure    0.009115
dtype: float64


In [15]:
with db.connect() as conn:
    po = pd.read_sql("SELECT patientid, outcome FROM diabetes.pedigree_outcome;", conn)

check = real.merge(po, on="patientid")
print(check.groupby(["measurement_date", "outcome"])[["glucose", "insulin", "bloodpressure"]].mean().round(1))

                          glucose  insulin  bloodpressure
measurement_date outcome                                 
2022-12-01       0          104.4     47.6           64.2
                 1           97.5     50.0           60.9
2022-12-13       0          110.0     68.8           68.2
                 1          141.3    100.3           70.8


In [16]:
query = """
SELECT *
FROM diabetes.patient p
LEFT JOIN diabetes.blood_metrics b ON p.id = b.patientid
LEFT JOIN diabetes.skin s ON p.id = s.patientid
LEFT JOIN diabetes.pedigree_outcome po ON p.id = po.patientid
WHERE b.measurement_date = '2022-12-13';
"""

with db.connect() as conn:
    df = pd.read_sql(query, conn)

print(df.shape)
df.head()

(768, 17)


,id,Age,pregnancies,bmi,insulin,glucose,bloodpressure,patientid,id,measurement_date,skinthickness,patientid,id,diabetespedigreefunction,outcome,patientid,id
0,1,50,6,33.6,0,148,72,1,1,2022-12-13,35,1,1,0.627,1,1,1
1,2,31,1,26.6,0,85,66,2,2,2022-12-13,29,2,2,0.351,0,2,2
2,3,32,8,23.3,0,183,64,3,3,2022-12-13,0,3,3,0.672,1,3,3
3,4,21,1,28.1,94,89,66,4,4,2022-12-13,23,4,4,0.167,0,4,4
4,5,33,0,43.1,168,137,40,5,5,2022-12-13,35,5,5,2.288,1,5,5


In [17]:
df = df.drop(columns=["id", "patientid", "measurement_date"])
df = df.rename(columns={
    "pregnancies": "Pregnancies",
    "glucose": "Glucose",
    "bloodpressure": "BloodPressure",
    "skinthickness": "SkinThickness",
    "insulin": "Insulin",
    "bmi": "BMI",
    "diabetespedigreefunction": "DiabetesPedigreeFunction",
    "outcome": "Outcome",
})
print(df.shape)
df.head()

(768, 9)


,Age,Pregnancies,BMI,Insulin,Glucose,BloodPressure,SkinThickness,DiabetesPedigreeFunction,Outcome
0,50,6,33.6,0,148,72,35,0.627,1
1,31,1,26.6,0,85,66,29,0.351,0
2,32,8,23.3,0,183,64,0,0.672,1
3,21,1,28.1,94,89,66,23,0.167,0
4,33,0,43.1,168,137,40,35,2.288,1


## Step 2 - Get the Data

The `diabetes` schema has four tables. All of them link to `patient` through `patient.id = patientid`.

| Table | Rows | Relationship to `patient` |
|---|---|---|
| `patient` | 768 | base table (age, pregnancies, BMI) |
| `skin` | 768 | 1-1 |
| `pedigree_outcome` | 768 | 1-1 (contains the target `outcome`) |
| `blood_metrics` | 1771 | 1-N (two measurement dates per patient) |

There are no N-M relationships.

### Findings

- **Orphan rows:** `blood_metrics` contains 1003 distinct `patientid` values, but only 768 exist in `patient`. The 235 extra ids have no age, BMI or outcome, so they are useless for modelling and are dropped by the join.
- **Two measurement dates:** each real patient has one blood row on `2022-12-01` and one on `2022-12-13`. The values differ for more than 99% of patients, so joining without a filter would duplicate every patient (1536 rows) and risk data leakage between train and test.
- **Which date is real?** I tested both dates against the outcome. On `2022-12-13`, diabetic patients have a much higher mean glucose than non-diabetic ones (141.3 vs 110.0), as expected medically. On `2022-12-01` the relation is reversed (97.5 vs 104.4), which suggests synthetic values. The `2022-12-13` values also match the original Kaggle dataset, so I keep only that date.

### Join choice

I use `LEFT JOIN` starting from `patient`, because the patient is the unit of analysis: every registered patient should stay in the data, and a missing measurement would show up as a missing value instead of silently removing the patient. In this data every patient has rows in all tables, so `LEFT JOIN` and `INNER JOIN` return the same 768 rows. Note that the `WHERE` filter on `measurement_date` would turn the `LEFT JOIN` into an `INNER JOIN` anyway.

After the join I drop the key columns (`id`, `patientid`) and `measurement_date`, and rename the columns to the original names. Result: 768 patients, 8 features and the target `Outcome`.

In [18]:
from sklearn.metrics import fbeta_score, make_scorer

f2_scorer = make_scorer(fbeta_score, beta=2)